In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

In [14]:
BASE = "https://books.toscrape.com/"

In [ ]:
def get_soup(url):
    try:
        resp = requests.get(url)
        resp.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        return BeautifulSoup(resp.text, "html.parser")
    except requests.exceptions.RequestException as e:
        print(f"Warning: Could not retrieve page {url}: {e}")
        return None


def parse_books_from_page(soup):
    books = []
    # Cada livro está dentro de <article class="product_pod">
    for artigo in soup.select("article.product_pod"):
        # título
        title = artigo.h3.a["title"]
        # preço
        price = artigo.select_one("p.price_color").text
        # disponibilidade
        availability = artigo.select_one("p.instock.availability").text.strip()
        # rating (class do parágrafo "star-rating X")
        rating_class = artigo.select_one("p.star-rating")["class"]
        # rating_class é lista, ex: ["star-rating", "Three"] — pegamos o segundo elemento
        rating = rating_class[1] if len(rating_class) > 1 else None
        # link para a página do livro (relativo)
        relative_url = artigo.h3.a["href"]
        book_url = urljoin(BASE, relative_url)
        # imagem (src relativo)
        img_rel = artigo.select_one("img")["src"]
        img_url = urljoin(BASE, img_rel)

        books.append({
            "title": title,
            "price": price,
            "availability": availability,
            "rating": rating,
            "book_url": book_url,
            "img_url": img_url
        })
    return books

def find_next_page(soup):
    # Verifica se há link "next"
    next_btn = soup.select_one("li.next > a")
    if next_btn:
        return urljoin(BASE, next_btn["href"])
    return None

def scrape_all_books(start_url):
    all_books = []
    url = start_url
    while url:
        soup = get_soup(url)
        if soup:  # Only proceed if soup was successfully created
            books = parse_books_from_page(soup)
            all_books.extend(books)
            url = find_next_page(soup)
        else:
            url = None # Stop if a page could not be retrieved
    return all_books

def clean_price(price_str):
    # price_str é algo tipo "£53.74"; queremos somente número float
    # removemos o símbolo “£” e convert
    return float(price_str.replace("£", "").replace("Â", "").strip())

def transform_data(raw_books):
    df = pd.DataFrame(raw_books)
    df["price"] = df["price"].map(clean_price)
    # transform rating textual para número (ex: One => 1, Two => 2 etc.)
    rating_map = {
        "One": 1, "Two": 2, "Three": 3,
        "Four": 4, "Five": 5
    }
    df["rating_num"] = df["rating"].map(rating_map)
    return df

def analyze(df):
    print("Quantidade de livros coletados:", len(df))
    print("Faixa de preços:")
    print(df["price"].describe())
    print("Distribuição de avaliações:")
    print(df["rating_num"].value_counts().sort_index())
    # Top 5 livros mais caros
    print("Top 5 mais caros:")
    print(df.nlargest(5, "price")[["title", "price", "rating"]])

def main():
    start = BASE  # pode ser "https://books.toscrape.com/index.html"
    raw = scrape_all_books(start)
    df = transform_data(raw)
    analyze(df)
    # salvar
    df.to_csv("books_data.csv", index=False, encoding="utf-8-sig")
    print("Dados salvos em books_data.csv")

if __name__ == "__main__":
    main()

Quantidade de livros coletados: 40
Faixa de preços:
count    40.000000
mean     34.958750
std      14.111908
min      12.840000
25%      22.575000
50%      34.080000
75%      50.407500
max      57.250000
Name: price, dtype: float64
Distribuição de avaliações:
rating_num
1     9
2     6
3     9
4     6
5    10
Name: count, dtype: int64
Top 5 mais caros:
                                                title  price rating
15  Our Band Could Be Your Life: Scenes from the A...  57.25  Three
25                      Birdsong: A Story in Pictures  54.64  Three
4               Sapiens: A Brief History of Humankind  54.23   Five
1                                  Tipping the Velvet  53.74    One
27                     Aladdin and His Wonderful Lamp  53.13  Three
Dados salvos em books_data.csv
